# Noise Measurement

## Instantiation

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep
from scipy import signal
from scipy.signal import decimate
from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qstl_instruments.qstl_nidaq import QSTL_NIDaq
%matplotlib inline

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)
t_acqu = 10 # acquisition time in second
decimation = 500
meas_seg = 6
total_meas_time = 3600 * 13
t = np.arange(start=0, stop=t_acqu, step=1/daq.max_sampling_rate)
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251117_NoiseData/noise.db")
exp = load_or_create_experiment("2D sweep", "noise_data")
meas = Measurement(exp=exp, station=station)

meas_time = Parameter(name = "meas_time", label = "Measurement Time", unit = "s")
meas_freq = Parameter(name = "meas_freq", label = "Measurement Frequency", unit = "Hz")
meas_rms = Parameter(name = "meas_rms", label = "Measurement RMS", unit = "pA")
trace_data = Parameter(name = "trace_data", label = "Measurement Data", unit = "pA")
fft_data = Parameter(name = "fft_data", label = "FFT Data", unit = "A/Hz^-1/2")
elapsed_time = Parameter(name = "elapsed_time", label = "Elapsed Time", unit = "s")
meas.register_parameter(meas_time)
meas.register_parameter(meas_freq)
meas.register_parameter(elapsed_time)
meas.register_parameter(fft_data, setpoints=(meas_freq, elapsed_time))
meas.register_parameter(trace_data, setpoints=(meas_time, elapsed_time))
meas.register_parameter(meas_rms, setpoints=(elapsed_time,))

## Measurement

In [14]:
start_time = time()

with meas.run() as datasaver:
    while True:
        num_of_samples = int(daq.max_sampling_rate * t_acqu)
        voltage_i = daq.read(
            ch = "Dev2/ai1",
            num_of_samples = num_of_samples
        )
        voltage_i = np.array(voltage_i)
        current_i = daq.convert_volts_to_amps(voltage_i)

        i_desired = decimate(current_i, decimation, ftype="fir", zero_phase=True)
        t_desired = np.linspace(0, t_acqu, len(i_desired))

        rms_pA = 1e12 * np.sqrt(np.mean((i_desired - np.mean((i_desired)))**2))
        print(f"RMS noise {rms_pA} pA")

        f, II_den = signal.periodogram(
            i_desired,
            fs = daq.max_sampling_rate/decimation,
            window = "flattop",
            scaling = "density",
            return_onesided = True
        )

        current_time = time() - start_time

        datasaver.add_result(
            (elapsed_time, [current_time] * len(f)),
            (meas_freq, f),
            (fft_data, np.sqrt(II_den))
        )
        datasaver.add_result(
            (elapsed_time, [current_time] * len(t_desired)),
            (meas_time, t_desired),
            (trace_data, 1e12 * i_desired)
        )
        datasaver.add_result(
            (elapsed_time, current_time),
            (meas_rms, rms_pA)
        )
        sleep((meas_seg - 1) * t_acqu)
        if current_time > total_meas_time:
            break

Starting experimental run with id: 6. 
RMS noise 3.307564232230049 pA
RMS noise 3.3185873890405877 pA
RMS noise 3.3462687451486595 pA
RMS noise 3.3864617445610796 pA
RMS noise 3.44128612043099 pA
RMS noise 3.3953691567825293 pA
RMS noise 3.3490201567863678 pA
RMS noise 5.038540497502778 pA
RMS noise 3.3675815786382306 pA
RMS noise 5.287705804432343 pA
RMS noise 5.053548920428264 pA
RMS noise 5.041765305569058 pA
RMS noise 3.30499879975433 pA
RMS noise 3.3624005559265386 pA
RMS noise 5.295380259706298 pA
RMS noise 3.332390898770567 pA
RMS noise 3.3532842763225323 pA
RMS noise 3.312989144615975 pA
RMS noise 3.4058727257747616 pA
RMS noise 3.3755044941244865 pA
RMS noise 3.3497508114497774 pA
RMS noise 3.381581684593268 pA
RMS noise 3.3794226896507973 pA
RMS noise 3.3528178723296467 pA
RMS noise 3.3901414557637164 pA
RMS noise 3.3153494154643557 pA
RMS noise 3.3998391194202062 pA
RMS noise 3.3181297121448554 pA
RMS noise 3.3486772538920135 pA
RMS noise 3.3781258230059152 pA
RMS noise 3.37